# 09 — CNN Training

Baselines to beat (notebook 08, same frozen split):
  slope-only LR      0.789
  handcrafted + SVM  0.826   <- best classical
  terrain only RF    0.788
  optical only RF    0.729

| Decision | Choice | Justification |
|---|---|---|
| Architecture | 4 conv blocks 32-64-128-256, BatchNorm | 64x64 input; 4 blocks give a 4x4 map before pooling. Deeper would over-downsample. |
| Head | global average pooling, not flatten | 256 params vs ~4k for flatten. Fewer parameters matter with only ~1,700 training samples. |
| Activation | ReLU | Standard; no vanishing gradient, cheap. |
| Loss | BCEWithLogits + pos_weight | Numerically stable; pos_weight corrects the 54.6% train prior. |
| Optimiser | AdamW, lr 1e-3, wd 1e-4 | Adaptive rates suit a small dataset; decoupled weight decay regularises better than L2-in-Adam. |
| Scheduler | cosine annealing | Smooth decay, no step tuning. |
| Batch size | 32 | Fits CPU memory; gradient noise aids generalisation. |
| Augmentation | flips, 90 deg rotations, optical-only brightness | Landslides have no canonical orientation. Hillshade encodes a fixed sun angle, so brightness jitter is applied to optical channels only. |
| Stopping | early stop on val PR-AUC, patience 15 | PR-AUC, not accuracy — the metric that matters here. |

Experiments: channel ablation, CNN vs MLP, transfer learning.

In [1]:
import json, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset

PROJECT_DIR = Path("..").resolve()
PROC_DIR    = PROJECT_DIR / "data" / "processed"
MODEL_DIR   = PROJECT_DIR / "outputs" / "models"
METRIC_DIR  = PROJECT_DIR / "outputs" / "metrics"
FIG_DIR     = PROJECT_DIR / "outputs" / "figures"
for d in (MODEL_DIR, METRIC_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}  device {DEVICE}")
if DEVICE == "cpu":
    print(f"CPU threads {torch.get_num_threads()} — expect ~20-40 s/epoch")

torch 2.13.0+cpu  device cpu
CPU threads 4 — expect ~20-40 s/epoch
